In [287]:
import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler   # u otros scalers
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet, LassoCV, RidgeCV, ElasticNetCV
from sklearn.metrics import mean_squared_error, r2_score
import haversine
from sklearn.preprocessing import RobustScaler
from sklearn.impute import KNNImputer

In [288]:
### carga datos de dataset en dataframe
file_path= 'uber_fares.csv'

df_copia = pd.read_csv(file_path)

df = df_copia
dfnan = df_copia

In [289]:
### visualizacion de algunos datos
df.head()

,key,date,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,24238194,2015-05-07 19:52:06.0000003,7.5,2015-05-07 19:52:06 UTC,-73.999817,40.738354,-73.999512,40.723217,1
1,27835199,2009-07-17 20:04:56.0000002,7.7,2009-07-17 20:04:56 UTC,-73.994355,40.728225,-73.994710,40.750325,1
2,44984355,2009-08-24 21:45:00.00000061,12.9,2009-08-24 21:45:00 UTC,-74.005043,40.740770,-73.962565,40.772647,1
3,25894730,2009-06-26 08:22:21.0000001,5.3,2009-06-26 08:22:21 UTC,-73.976124,40.790844,-73.965316,40.803349,3
4,17610152,2014-08-28 17:47:00.000000188,16.0,2014-08-28 17:47:00 UTC,-73.925023,40.744085,-73.973082,40.761247,5


#### Contexto  
El proyecto trata sobre **Uber Inc.**, la compañía de taxis más grande del mundo. En este trabajo, nuestro objetivo es **predecir la tarifa de futuros viajes**.  

Uber brinda servicio a millones de clientes cada día, por lo que gestionar adecuadamente sus datos es clave para desarrollar nuevas estrategias de negocio y obtener mejores resultados.  

### Variables del conjunto de datos  

**Variables explicativas:**  
- **key**: identificador único de cada viaje.  
- **pickup_datetime**: fecha y hora en que se inició el viaje.  
- **passenger_count**: cantidad de pasajeros en el vehículo (dato ingresado por el conductor).  
- **pickup_longitude**: longitud del punto de inicio del viaje.  
- **pickup_latitude**: latitud del punto de inicio del viaje.  
- **dropoff_longitude**: longitud del punto de destino.  
- **dropoff_latitude**: latitud del punto de destino.  

**Variable objetivo (target):**  
- **fare_amount**: costo del viaje en dólares.  

In [290]:
### Columnas, ¿cuáles son variables numéricas y cuales variables categóricas?
df.columns

Index(['key', 'date', 'fare_amount', 'pickup_datetime', 'pickup_longitude',
       'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude',
       'passenger_count'],
      dtype='object')

In [291]:
df.describe(include='all')

,key,date,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
count,2.000000e+05,200000,200000.000000,200000,200000.000000,200000.000000,199999.000000,199999.000000,200000.000000
unique,NaN,200000,NaN,196629,NaN,NaN,NaN,NaN,NaN
top,NaN,2015-05-07 19:52:06.0000003,NaN,2009-02-12 12:46:00 UTC,NaN,NaN,NaN,NaN,NaN
freq,NaN,1,NaN,4,NaN,NaN,NaN,NaN,NaN
mean,2.771250e+07,NaN,11.359955,NaN,-72.527638,39.935885,-72.525292,39.923890,1.684535
std,1.601382e+07,NaN,9.901776,NaN,11.437787,7.720539,13.117408,6.794829,1.385997
min,1.000000e+00,NaN,-52.000000,NaN,-1340.648410,-74.015515,-3356.666300,-881.985513,0.000000
25%,1.382535e+07,NaN,6.000000,NaN,-73.992065,40.734796,-73.991407,40.733823,1.000000
50%,2.774550e+07,NaN,8.500000,NaN,-73.981823,40.752592,-73.980093,40.753042,1.000000
75%,4.155530e+07,NaN,12.500000,NaN,-73.967154,40.767158,-73.963658,40.768001,2.000000


In [292]:
print(sum(df['pickup_longitude'] > 180)) # valores de longitud y latitud por fuera de (-180 - 180) y (-90 - 90), respectivamente
print(sum(df['dropoff_latitude'] < -90))
print(sum(df['dropoff_longitude'] < -180))
print(sum(df['pickup_latitude'] > 90))
print(sum(df['pickup_latitude'] == 0))
print(sum(df['pickup_longitude'] == 0))
print(sum(df['dropoff_latitude'] == 0))
print(sum(df['dropoff_longitude'] == 0))
print(sum((df['pickup_longitude'] == 0)& (df['dropoff_latitude'] == 0)& (df['dropoff_longitude'] == 0) & (df['pickup_latitude'] == 0)))
print(sum(df['fare_amount']==0))

0
1
2
4
3782
3786
3758
3764
3587
5


In [293]:
# REEMPLAZO DE 0 DE LAS COLUMNAS DE COORDENADAS POR NAN, PARA EL ESCALADO

float_cols = ['pickup_longitude','pickup_latitude','dropoff_longitude','dropoff_latitude']
for col in float_cols:
    df.loc[df[col] == 0, col] = np.nan



In [294]:
# borrado de los valores por fuera de estos valores, representan un % insignificante, MANTENIENDO LOS NAN

df = df[df['dropoff_latitude'].between(-90, 90) | df['dropoff_latitude'].isna()] 
df = df[df['pickup_longitude'].between(-180, 180) | df['pickup_longitude'].isna()]
df = df[df['pickup_latitude'].between(-90, 90)| df['pickup_latitude'].isna()]
df = df[df['dropoff_longitude'].between(-180, 180) | df['dropoff_longitude'].isna()]

In [295]:
print(sum(df['dropoff_longitude'] == df["pickup_longitude"]))
print(sum(df['dropoff_latitude'] == df["pickup_latitude"]))
# TENEMOS QUE VER QUÉ HACEMOS CON LOS QUE TIENEN PUNTO DE SALIDA Y RETORNO IGUALES

2081
2090


In [296]:
df.columns[df.isna().any()] 

Index(['pickup_longitude', 'pickup_latitude', 'dropoff_longitude',
       'dropoff_latitude'],
      dtype='object')

In [297]:
# % de nan en cada columna
(df.isna().sum() / len(df)) * 100

key                  0.000000
date                 0.000000
fare_amount          0.000000
pickup_datetime      0.000000
pickup_longitude     1.893114
pickup_latitude      1.891113
dropoff_longitude    1.882613
dropoff_latitude     1.879613
passenger_count      0.000000
dtype: float64

In [298]:
df_limpio = df

In [299]:
df_limpio.info()
df_limpio.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
Index: 199988 entries, 0 to 199999
Data columns (total 9 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   key                199988 non-null  int64  
 1   date               199988 non-null  object 
 2   fare_amount        199988 non-null  float64
 3   pickup_datetime    199988 non-null  object 
 4   pickup_longitude   196202 non-null  float64
 5   pickup_latitude    196206 non-null  float64
 6   dropoff_longitude  196223 non-null  float64
 7   dropoff_latitude   196229 non-null  float64
 8   passenger_count    199988 non-null  int64  
dtypes: float64(5), int64(2), object(2)
memory usage: 15.3+ MB


key                     0
date                    0
fare_amount             0
pickup_datetime         0
pickup_longitude     3786
pickup_latitude      3782
dropoff_longitude    3765
dropoff_latitude     3759
passenger_count         0
dtype: int64

In [300]:
X_train, X_test, y_train, y_test = train_test_split(df_limpio.drop(columns='fare_amount'), df_limpio['fare_amount'], test_size=0.2, random_state=42)

In [301]:
X_train.shape

(159990, 8)

In [302]:
X_train.isna().sum()

key                     0
date                    0
pickup_datetime         0
pickup_longitude     3073
pickup_latitude      3069
dropoff_longitude    3064
dropoff_latitude     3058
passenger_count         0
dtype: int64

In [303]:
X_train.describe(include='all')

,key,date,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
count,1.599900e+05,159990,159990,156917.000000,156921.000000,156926.000000,156932.000000,159990.000000
unique,NaN,159990,157838,NaN,NaN,NaN,NaN,NaN
top,NaN,2015-01-06 16:30:04.0000002,2014-04-13 18:19:00 UTC,NaN,NaN,NaN,NaN,NaN
freq,NaN,1,4,NaN,NaN,NaN,NaN,NaN
mean,2.772714e+07,NaN,NaN,-73.900756,40.687302,-73.902351,40.687059,1.679505
std,1.602099e+07,NaN,NaN,2.812751,2.640015,2.774024,2.642838,1.303397
min,1.000000e+00,NaN,NaN,-93.824668,-74.015515,-75.458979,-74.015750,0.000000
25%,1.383602e+07,NaN,NaN,-73.992281,40.736365,-73.991596,40.735192,1.000000
50%,2.776453e+07,NaN,NaN,-73.982102,40.753272,-73.980532,40.753728,1.000000
75%,4.158793e+07,NaN,NaN,-73.968295,40.767560,-73.965273,40.768331,2.000000


In [304]:
X_train['day_week_num'] = pd.to_datetime(X_train['date'], utc=True) # convierte a datetime
X_train['day_week_num'] = X_train['day_week_num'].dt.dayofweek # funcón para sacar el día de la semana (numérico)
X_train['day_week_num']

48971     1
131208    5
15045     4
11426     3
49247     0
         ..
119887    3
103701    4
131940    6
146877    5
121966    3
Name: day_week_num, Length: 159990, dtype: int32

In [305]:
# Función Circular encoding

def circular_encoding(X_train, column, n_categorias):
    sin_col = f"{column}_sin"
    cos_col = f"{column}_cos"
    X_train[sin_col] = np.sin((2 * np.pi * (X_train[column]))/ n_categorias)
    X_train[cos_col] = np.cos((2 * np.pi * (X_train[column])) / n_categorias)
    return X_train

X_train = circular_encoding(X_train, 'day_week_num', 7)

X_train['day_week_num_sin']


48971     0.781831
131208   -0.974928
15045    -0.433884
11426     0.433884
49247     0.000000
            ...   
119887    0.433884
103701   -0.433884
131940   -0.781831
146877   -0.974928
121966    0.433884
Name: day_week_num_sin, Length: 159990, dtype: float64

In [306]:
# Circular enconding para el momento del día

X_train['fecha_hora'] = pd.to_datetime(X_train['date'].str.replace(' UTC', ''), utc=True) #convertirmos a datetime

X_train['hora'] = X_train['fecha_hora'].dt.hour # extraemos la hora de la fecha general

X_train = circular_encoding(X_train, 'hora', 24)

X_train['hora_cos']

48971    -0.500000
131208   -0.707107
15045    -0.707107
11426     0.965926
49247     0.500000
            ...   
119887   -0.500000
103701   -0.866025
131940   -0.866025
146877    0.707107
121966    0.707107
Name: hora_cos, Length: 159990, dtype: float64

In [307]:
# ESCALADO ROBUSTO

scaler = RobustScaler()
cols_a_escalar = ['pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'passenger_count']

# Identifica las filas sin NaN
filas_sin_nan = ~X_train[cols_a_escalar].isna().any(axis=1)
df_sin_nan = X_train.loc[filas_sin_nan, cols_a_escalar]

#  Escala sólo las filas sin NaN
df_sin_nan_escalado_array = scaler.fit_transform(df_sin_nan)

#  Crea un nuevo DataFrame con los datos escalados, manteniendo el índice original
columnas_escaladas = [col + '_escalada' for col in cols_a_escalar]
df_sin_nan_escalado = pd.DataFrame(df_sin_nan_escalado_array, columns=columnas_escaladas, index=df_sin_nan.index)

# Une las nuevas columnas escaladas al DataFrame X_train original
X_train[columnas_escaladas] = np.nan

#  Rellena las nuevas columnas con los valores escalados
X_train.loc[df_sin_nan_escalado.index, columnas_escaladas] = df_sin_nan_escalado.values

In [308]:
print(X_train[cols_a_escalar].dtypes)
print(X_train[cols_a_escalar].isna().sum())

pickup_longitude     float64
pickup_latitude      float64
dropoff_longitude    float64
dropoff_latitude     float64
passenger_count        int64
dtype: object
pickup_longitude     3073
pickup_latitude      3069
dropoff_longitude    3064
dropoff_latitude     3058
passenger_count         0
dtype: int64


In [309]:
X_train.describe(include="all")

,key,date,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,day_week_num,day_week_num_sin,day_week_num_cos,fecha_hora,hora,hora_sin,hora_cos,pickup_longitude_escalada,pickup_latitude_escalada,dropoff_longitude_escalada,dropoff_latitude_escalada,passenger_count_escalada
count,1.599900e+05,159990,159990,156917.000000,156921.000000,156926.000000,156932.000000,159990.000000,159990.000000,159990.000000,159990.000000,159990,159990.000000,159990.000000,159990.000000,156763.000000,156763.000000,156763.000000,156763.000000,156763.000000
unique,NaN,159990,157838,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,2015-01-06 16:30:04.0000002,2014-04-13 18:19:00 UTC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,1,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,2.772714e+07,NaN,NaN,-73.900756,40.687302,-73.902351,40.687059,1.679505,3.048622,-0.000689,-0.043768,2012-03-23 10:17:36.764647936+00:00,13.495550,-0.195137,-0.068667,3.257256,-2.117045,2.938510,-2.005600,0.680377
min,1.000000e+00,NaN,NaN,-93.824668,-74.015515,-75.458979,-74.015750,0.000000,0.000000,-0.974928,-0.900969,2009-01-01 01:15:22.000000600+00:00,0.000000,-1.000000,-1.000000,-827.428589,-3679.607316,-56.196310,-3463.536644,-1.000000
25%,1.383602e+07,NaN,NaN,-73.992281,40.736365,-73.991596,40.735192,1.000000,1.000000,-0.781831,-0.900969,2010-08-26 20:59:30.000000256+00:00,9.000000,-0.866025,-0.707107,-0.424419,-0.542072,-0.420358,-0.559413,0.000000
50%,2.776453e+07,NaN,NaN,-73.982102,40.753272,-73.980532,40.753728,1.000000,3.000000,0.000000,-0.222521,2012-03-23 14:58:31.000000512+00:00,14.000000,-0.258819,-0.258819,0.000000,0.000000,0.000000,0.000000,0.000000
75%,4.158793e+07,NaN,NaN,-73.968295,40.767560,-73.965273,40.768331,2.000000,5.000000,0.781831,0.623490,2013-10-18 21:36:02.750000128+00:00,19.000000,0.500000,0.707107,0.575581,0.457928,0.579642,0.440587,1.000000
max,5.542357e+07,NaN,NaN,40.808425,47.383332,40.831932,45.031598,6.000000,6.000000,0.974928,1.000000,2015-06-30 23:40:39.000000100+00:00,23.000000,1.000000,1.000000,4786.728160,212.566487,4364.082711,129.098305,5.000000


In [310]:
# IMPUTACIÓN CON KNN A LAS COORDENDAS

imputer = KNNImputer(n_neighbors=5)

cols = ['pickup_longitude_escalada','pickup_latitude_escalada','dropoff_latitude_escalada','dropoff_longitude_escalada']

X_train[cols] = imputer.fit_transform(X_train[cols])

In [311]:
print(X_train[cols].dtypes)
print(X_train[cols].isna().sum())


pickup_longitude_escalada     float64
pickup_latitude_escalada      float64
dropoff_latitude_escalada     float64
dropoff_longitude_escalada    float64
dtype: object
pickup_longitude_escalada     0
pickup_latitude_escalada      0
dropoff_latitude_escalada     0
dropoff_longitude_escalada    0
dtype: int64


In [312]:
col_escaladas = ['pickup_longitude_escalada', 'pickup_latitude_escalada', 'dropoff_latitude_escalada', 'dropoff_longitude_escalada', 'passenger_count_escalada']

# Desescala los datos de TODAS las columnas
datos_desescalados_array = scaler.inverse_transform(X_train[col_escaladas].values)

# Crea una lista con los nombres de las nuevas columnas con el sufijo '_desescalada'
col_desescaladas = [col.replace('_escalada', '_desescalada') for col in col_escaladas]

# 3. Asigna los valores desescalados a las NUEVAS columnas en X_train
X_train[col_desescaladas] = datos_desescalados_array

In [313]:
X_train.describe(include="all")

,key,date,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,day_week_num,day_week_num_sin,...,pickup_longitude_escalada,pickup_latitude_escalada,dropoff_longitude_escalada,dropoff_latitude_escalada,passenger_count_escalada,pickup_longitude_desescalada,pickup_latitude_desescalada,dropoff_latitude_desescalada,dropoff_longitude_desescalada,passenger_count_desescalada
count,1.599900e+05,159990,159990,156917.000000,156921.000000,156926.000000,156932.000000,159990.000000,159990.000000,159990.000000,...,159990.000000,159990.000000,159990.000000,159990.000000,156763.000000,159990.000000,159990.000000,159990.000000,159990.000000,156763.000000
unique,NaN,159990,157838,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,2015-01-06 16:30:04.0000002,2014-04-13 18:19:00 UTC,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,1,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,2.772714e+07,NaN,NaN,-73.900756,40.687302,-73.902351,40.687059,1.679505,3.048622,-0.000689,...,3.257256,-2.117045,2.938510,-2.005600,0.680377,-73.903991,40.687245,-74.033302,40.851104,1.680377
min,1.000000e+00,NaN,NaN,-93.824668,-74.015515,-75.458979,-74.015750,0.000000,0.000000,-0.974928,...,-827.428589,-3679.607316,-56.196310,-3463.536644,-1.000000,-93.824668,-74.015515,-165.100992,38.891583,0.000000
25%,1.383602e+07,NaN,NaN,-73.992281,40.736365,-73.991596,40.735192,1.000000,1.000000,-0.781831,...,-0.415412,-0.595782,-0.412709,-0.606544,0.000000,-73.992065,40.734694,-73.996495,40.740056,1.000000
50%,2.776453e+07,NaN,NaN,-73.982102,40.753272,-73.980532,40.753728,1.000000,3.000000,0.000000,...,0.012552,-0.023324,0.017143,-0.022437,0.000000,-73.981802,40.752550,-73.981128,40.754300,1.000000
75%,4.158793e+07,NaN,NaN,-73.968295,40.767560,-73.965273,40.768331,2.000000,5.000000,0.781831,...,0.627747,0.444623,0.645827,0.429888,1.000000,-73.967049,40.767145,-73.969228,40.775132,2.000000
max,5.542357e+07,NaN,NaN,40.808425,47.383332,40.831932,45.031598,6.000000,6.000000,0.974928,...,4786.728160,212.566487,4364.082711,129.098305,5.000000,40.808425,47.383332,-70.584155,185.364159,6.000000


In [317]:
print(sum(X_train['pickup_longitude'] > 180)) # valores de longitud y latitud por fuera de (-180 - 180) y (-90 - 90), respectivamente
print(sum(X_train['dropoff_latitude'] < -90))
print(sum(X_train['dropoff_longitude'] < -180))
print(sum(X_train['pickup_latitude'] > 90))
(sum(X_train['dropoff_latitude'] < -90)) / len(X_train) *100

0
0
0
0


0.0

In [315]:
# los datos por fuera de estos valores representan el 0.06%, en consecuencia los borramos

X_train = X_train[X_train['dropoff_latitude'].between(-90, 90)]

In [316]:
# Cálculo de distancia recorrida en km según latitud y longitud(inicio y destino)
X_train['km_distance'] = X_train.apply(lambda row: haversine.haversine((row['pickup_latitude'], row['pickup_longitude']),(row['dropoff_latitude'], row['dropoff_longitude'])), axis=1)
X_train['km_distance'].describe()

count    156763.000000
mean          4.396867
std          76.812346
min           0.000000
25%           1.253727
50%           2.155273
75%           3.905447
max        8667.705150
Name: km_distance, dtype: float64